# Paragliding Weather Alert for Beginners

Tento notebook automaticky hlídá počasí na oficiálních vzletových plochách v okruhu 200 km od Českých Budějovic a upozorní tě e-mailem, pokud budou zítra vhodné podmínky pro začátečníka. 

**Funkce:**
- Hlídání větru, nárazů, srážek, směru a synoptické situace
- Automatické vyhodnocení a report
- Odeslání e-mailu při vhodných podmínkách
- Možnost naplánovat automatické spouštění

---

## 1. Import potřebných knihoven

In [ ]:
# Import knihoven
import requests
import pandas as pd
import numpy as np
import math
import datetime
import smtplib
import os
import json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import logging
# Pro plánování (vyber jednu z nich, podle prostředí)
try:
    import schedule
except ImportError:
    schedule = None
try:
    from apscheduler.schedulers.background import BackgroundScheduler
except ImportError:
    BackgroundScheduler = None


## 2. Definice vzletových ploch v okruhu 200 km od Českých Budějovic

Seznam obsahuje jméno, souřadnice, nadmořskou výšku, povolené směry větru (azimut, rozsah v °), a obtížnost. Vzletové plochy jsou filtrovány podle vzdálenosti (haversinova formule).

In [ ]:
# Souřadnice Českých Budějovic
CB_LAT, CB_LON = 48.9747, 14.4743

# Vzletové plochy (ukázkový výběr, doplň další podle potřeby)
sites = [
    {"name": "Kleť", "lat": 48.8581, "lon": 14.2831, "elev": 1084, "azimuth_min": 270, "azimuth_max": 360, "difficulty": "easy"},
    {"name": "Kozí Pláň", "lat": 48.7267, "lon": 14.1631, "elev": 885, "azimuth_min": 90, "azimuth_max": 180, "difficulty": "easy"},
    {"name": "Javorový vrch", "lat": 49.6517, "lon": 18.6261, "elev": 1032, "azimuth_min": 0, "azimuth_max": 120, "difficulty": "medium"},
    {"name": "Raná", "lat": 50.3972, "lon": 13.8031, "elev": 457, "azimuth_min": 270, "azimuth_max": 90, "difficulty": "easy"},
    {"name": "Dürrnberg (A)", "lat": 47.6350, "lon": 13.0930, "elev": 1230, "azimuth_min": 180, "azimuth_max": 270, "difficulty": "easy"},
    # ...další plochy...
]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*R*math.asin(math.sqrt(a))

for s in sites:
    s["distance_km"] = haversine(CB_LAT, CB_LON, s["lat"], s["lon"])

sites_df = pd.DataFrame([s for s in sites if s["distance_km"] <= 200])
sites_df = sites_df.sort_values("distance_km").reset_index(drop=True)
sites_df

## 3. Definice podmínek vhodných pro začátečníka

Nastavíme limity pro bezpečné létání: maximální průměrná rychlost větru, maximální náraz, poměr náraz/průměr, nulové srážky, tolerance směru větru a stabilní tlak. Vše uložíme do konfiguračního slovníku.

In [ ]:
# Konfigurační limity pro začátečníka
BEGINNER_LIMITS = {
    "max_wind_avg": 20,        # km/h
    "max_wind_gust": 25,      # km/h
    "max_gust_factor": 1.4,   # poměr náraz/průměr
    "precipitation": 0.0,     # mm
    "direction_tolerance": 45, # stupňů od povoleného směru
    "min_flyable_hours": 3,   # min. počet po sobě jdoucích hodin
    "daylight_hours": (9, 18) # 9:00-18:00
}


## 4. Stažení předpovědi počasí pro zítřek

Použijeme Open-Meteo API (zdarma, bez klíče) pro každou plochu. Stáhneme hodinovou předpověď: rychlost a směr větru, nárazy, srážky, tlak, oblačnost.

In [ ]:
def fetch_weather(lat, lon):
    # Open-Meteo API endpoint
    url = (
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}"
        "&hourly=wind_speed_10m,wind_gusts_10m,wind_direction_10m,precipitation,pressure_msl,cloudcover"
        "&forecast_days=2&timezone=Europe/Prague"
    )
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

def get_tomorrow_hours():
    now = datetime.datetime.now()
    tomorrow = now + datetime.timedelta(days=1)
    return [f"{tomorrow.strftime('%Y-%m-%d')}T{h:02d}:00" for h in range(24)]

weather_data = {}
for idx, row in sites_df.iterrows():
    site_name = row['name']
    w = fetch_weather(row['lat'], row['lon'])
    hours = get_tomorrow_hours()
    df = pd.DataFrame({
        'time': w['hourly']['time'],
        'wind_speed': w['hourly']['wind_speed_10m'],
        'wind_gust': w['hourly']['wind_gusts_10m'],
        'wind_dir': w['hourly']['wind_direction_10m'],
        'precip': w['hourly']['precipitation'],
        'pressure': w['hourly']['pressure_msl'],
        'cloud': w['hourly']['cloudcover'],
    })
    df = df[df['time'].isin(hours)].reset_index(drop=True)
    weather_data[site_name] = df

# Ukázka pro první plochu
display(weather_data[list(weather_data.keys())[0]].head())

## 5. Vyhodnocení směru větru vůči orientaci plochy

Pro každou hodinu a plochu ověříme, zda směr větru spadá do povoleného azimutového okna. Funkce správně ošetří přechod přes 0/360°.

In [ ]:
def wind_in_window(wind_dir, az_min, az_max, tol):
    # Ošetření přechodu přes 0/360°
    az_min = (az_min - tol) % 360
    az_max = (az_max + tol) % 360
    if az_min < az_max:
        return (wind_dir >= az_min) & (wind_dir <= az_max)
    else:
        return (wind_dir >= az_min) | (wind_dir <= az_max)

# Přidáme sloupec s informací, zda je směr větru v okně
for idx, row in sites_df.iterrows():
    site = row['name']
    az_min, az_max = row['azimuth_min'], row['azimuth_max']
    tol = BEGINNER_LIMITS['direction_tolerance']
    weather_data[site]['wind_ok'] = weather_data[site]['wind_dir'].apply(lambda wd: wind_in_window(wd, az_min, az_max, tol))

# Ukázka
weather_data[list(weather_data.keys())[0]][['time', 'wind_dir', 'wind_ok']].head()

## 6. Vyhodnocení rychlosti větru, nárazů a srážek

Aplikujeme limity pro začátečníka a vytvoříme sloupec 'flyable', kde jsou všechny podmínky splněny.

In [ ]:
for idx, row in sites_df.iterrows():
    site = row['name']
    df = weather_data[site]
    df['gust_factor'] = df['wind_gust'] / df['wind_speed'].replace(0, np.nan)
    df['wind_speed_ok'] = df['wind_speed'] <= BEGINNER_LIMITS['max_wind_avg']
    df['wind_gust_ok'] = df['wind_gust'] <= BEGINNER_LIMITS['max_wind_gust']
    df['gust_factor_ok'] = df['gust_factor'] <= BEGINNER_LIMITS['max_gust_factor']
    df['precip_ok'] = df['precip'] <= BEGINNER_LIMITS['precipitation']
    df['flyable'] = df['wind_ok'] & df['wind_speed_ok'] & df['wind_gust_ok'] & df['gust_factor_ok'] & df['precip_ok']
    weather_data[site] = df

# Ukázka
weather_data[list(weather_data.keys())[0]][['time', 'flyable']].head()

## 7. Analýza synoptického (baryckého) vývoje počasí

Stáhneme regionální předpověď tlaku a geopotenciálu, vyhodnotíme tendenci a klasifikujeme synoptickou situaci (stabilní, fronta, po frontě).

In [ ]:
def fetch_synoptic(lat=49.5, lon=15.5):
    url = (
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}"
        "&hourly=pressure_msl,geopotential_height_500hPa"
        "&forecast_days=2&timezone=Europe/Prague"
    )
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

def analyze_synoptic(syn_data):
    # Zjednodušená analýza: změna tlaku za 24h
    p = np.array(syn_data['hourly']['pressure_msl'])
    t = syn_data['hourly']['time']
    now = datetime.datetime.now()
    tomorrow = now + datetime.timedelta(days=1)
    idx_now = [i for i, ti in enumerate(t) if ti.startswith(now.strftime('%Y-%m-%d'))]
    idx_tom = [i for i, ti in enumerate(t) if ti.startswith(tomorrow.strftime('%Y-%m-%d'))]
    if not idx_now or not idx_tom:
        return 'unknown', 0
    p_now = np.mean(p[idx_now])
    p_tom = np.mean(p[idx_tom])
    delta = p_tom - p_now
    if delta > 2:
        return 'stabilní vysoký tlak', delta
    elif delta < -2:
        return 'fronta / pokles tlaku', delta
    else:
        return 'mírně proměnlivé', delta

synoptic = fetch_synoptic()
synoptic_status, synoptic_delta = analyze_synoptic(synoptic)
print(f"Synoptická situace: {synoptic_status} (změna tlaku {synoptic_delta:.1f} hPa)")

## 8. Výběr a skórování ploch vhodných na zítřek

Spočítáme počet letuschopných hodin v denním okně, přidáme synoptické skóre a doporučíme pouze plochy s dostatečným počtem po sobě jdoucích hodin.

In [ ]:
def find_flyable_windows(df, min_hours, daylight=(9, 18)):
    # Filtrujeme na denní hodiny
    df['hour'] = df['time'].str[11:13].astype(int)
    day_df = df[(df['hour'] >= daylight[0]) & (df['hour'] <= daylight[1])]
    # Najdeme maximální počet po sobě jdoucích letuschopných hodin
    flyable = day_df['flyable'].values
    max_streak = 0
    streak = 0
    for ok in flyable:
        if ok:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak, day_df

results = []
for idx, row in sites_df.iterrows():
    site = row['name']
    streak, day_df = find_flyable_windows(weather_data[site], BEGINNER_LIMITS['min_flyable_hours'], BEGINNER_LIMITS['daylight_hours'])
    results.append({
        'site': site,
        'max_streak': streak,
        'flyable_hours': int(day_df['flyable'].sum()),
        'distance_km': row['distance_km'],
        'difficulty': row['difficulty'],
    })
results_df = pd.DataFrame(results)
results_df = results_df[results_df['max_streak'] >= BEGINNER_LIMITS['min_flyable_hours']]
results_df = results_df.sort_values(['max_streak', 'flyable_hours', 'distance_km'], ascending=[False, False, True]).reset_index(drop=True)
results_df

## 9. Generování lidsky čitelného reportu

Vytvoříme textový/HTML report s doporučenými plochami, časovými okny, očekávaným větrem, srážkami a synoptickou situací. Přidáme verdikt "letět / neletět".

In [ ]:
def build_report(results_df, weather_data, synoptic_status, synoptic_delta):
    if results_df.empty:
        return "<b>Zítra nejsou vhodné podmínky pro začátečníka na žádné sledované ploše.</b>"
    report = f"<h2>Doporučené plochy pro zítřek</h2>"
    report += f"<p>Synoptická situace: <b>{synoptic_status}</b> (změna tlaku {synoptic_delta:.1f} hPa)</p>"
    for _, row in results_df.iterrows():
        site = row['site']
        df = weather_data[site]
        fly_hours = df[df['flyable']]
        if fly_hours.empty:
            continue
        t1, t2 = fly_hours['time'].iloc[0][11:16], fly_hours['time'].iloc[-1][11:16]
        wind = fly_hours['wind_speed'].mean()
        gust = fly_hours['wind_gust'].mean()
        wind_dir = fly_hours['wind_dir'].mean()
        report += f"<h3>{site} ({row['distance_km']:.0f} km, {row['difficulty']})</h3>"
        report += f"<ul>"
        report += f"<li>Letuschopné okno: {t1} - {t2}</li>"
        report += f"<li>Průměrný vítr: {wind:.1f} km/h, nárazy: {gust:.1f} km/h, směr: {wind_dir:.0f}°</li>"
        report += f"<li>Počet letuschopných hodin: {row['flyable_hours']}</li>"
        report += f"<li>Verdikt: <b>LETĚT!</b></li>"
        report += f"</ul>"
    return report

report_html = build_report(results_df, weather_data, synoptic_status, synoptic_delta)
from IPython.display import display, HTML
display(HTML(report_html))

## 10. Odeslání e-mailu s upozorněním

Použijeme SMTP přes SSL, přihlašovací údaje načteme z proměnných prostředí. E-mail odešleme pouze pokud je aspoň jedna plocha vhodná.

In [ ]:
def send_email(report_html):
    smtp_server = os.environ.get('SMTP_SERVER')
    smtp_port = int(os.environ.get('SMTP_PORT', 465))
    smtp_user = os.environ.get('SMTP_USER')
    smtp_pass = os.environ.get('SMTP_PASS')
    to_addr = os.environ.get('ALERT_EMAIL')
    if not all([smtp_server, smtp_user, smtp_pass, to_addr]):
        print("Chybí SMTP údaje v proměnných prostředí!")
        return
    msg = MIMEMultipart('alternative')
    msg['Subject'] = "Paragliding Alert: Vhodné podmínky na zítřek"
    msg['From'] = smtp_user
    msg['To'] = to_addr
    part = MIMEText(report_html, 'html')
    msg.attach(part)
    with smtplib.SMTP_SSL(smtp_server, smtp_port) as server:
        server.login(smtp_user, smtp_pass)
        server.sendmail(smtp_user, to_addr, msg.as_string())
    print(f"E-mail odeslán na {to_addr}")

if not results_df.empty:
    send_email(report_html)

## 11. Automatizace: plánování pravidelného běhu

Použijeme knihovnu `schedule` nebo `APScheduler` pro automatické spouštění každou hodinu. Logujeme do souboru.

In [ ]:
def run_check():
    logging.basicConfig(filename='paragliding_alert.log', level=logging.INFO,
                        format='%(asctime)s %(levelname)s %(message)s')
    try:
        # --- Zde by se volaly všechny kroky pipeline ---
        # Pro zjednodušení zde pouze logujeme
        logging.info('Kontrola počasí spuštěna.')
        # ... zde by se volal hlavní kód ...
    except Exception as e:
        logging.error(f'Chyba: {e}')

# Automatizace pomocí schedule nebo APScheduler
if schedule:
    schedule.every().hour.do(run_check)
    print('Spouštím plánovač (schedule)...')
    while True:
        schedule.run_pending()
        time.sleep(60)
elif BackgroundScheduler:
    scheduler = BackgroundScheduler()
    scheduler.add_job(run_check, 'interval', hours=1)
    scheduler.start()
    print('Spouštím plánovač (APScheduler)...')
    import time
    try:
        while True:
            time.sleep(60)
    except (KeyboardInterrupt, SystemExit):
        scheduler.shutdown()
else:
    print('Není dostupná knihovna pro plánování. Spusť ručně nebo doinstaluj schedule/APScheduler.')

## 12. Příprava GIT repozitáře pro jiný GitHub účet

Shell příkazy pro inicializaci repozitáře, nastavení jiného uživatele, přidání vzdáleného repa a push. Přidáme .gitignore a README.

In [ ]:
# Bash příkazy pro nastavení repozitáře (spouštěj v shellu, ne v notebooku)
print('''
# Inicializace repozitáře
git init
# Nastavení uživatele pro tento repozitář
git config user.name "TvojeJmeno"
git config user.email "tvoje@email.cz"
# Přidání vzdáleného repozitáře (nahraď URL svým repem)
git remote add origin git@github.com:tvujnovyucet/paragliding-weather-alert.git
# Přidání .gitignore a README
(echo "*.env\n__pycache__/\nparagliding_alert.log" > .gitignore)
echo "# Paragliding Weather Alert" > README.md
# První commit a push
git add .
git commit -m "Initial commit"
git push -u origin master
''')